In [3]:
import numpy as np
import matplotlib.pyplot as plt
import os
import astropy as ast
from astropy.io import ascii
import pandas as pd
import matplotlib.cm as cm
from matplotlib.colors import LogNorm, Normalize
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
#extinction imports
from astropy.table import Table, Column, MaskedColumn, vstack
from astropy import units as u
from astropy.coordinates import SkyCoord
# from zh_scraper import get_extinction
# from dustmaps.sfd import SFDQuery
# from dustmaps.edenhofer2023 import Edenhofer2023Query
# sfd = SFDQuery()
# eden = Edenhofer2023Query(integrated=True)
from view_and_clean import offset_corrector #individual_plotter, df_extract, individual_plotter_v2, df_extract_v2,
from metrics import sed_plotter, info, observed_sed, largest_amplitude #, fit_plotter_flux, fit_plotter_flux_v2, mean_med_flux, mean_med_flux_v2, offset_warning, offset_warning_v2,compute_lomb_scargle, plot_periodogram, plot_phase_fold 

In [6]:
# for ease of access I created this file that contains RA and DEC columns of both SMC and LMC YSG candidates combined:
coords = pd.read_csv('../merged_smc_lmc_coords.csv', comment='#', sep="\\s+", names=['RA', 'DEC'])

df_lmc = pd.read_csv('../ysg_candidates/original_files_from_anna/final_lmc_ysgcands.csv', comment='#') # , sep="\\s+"
df_smc = pd.read_csv('../ysg_candidates/original_files_from_anna/final_smc_ysgcands.csv', comment='#') # , sep="\\s+"

# now Anna's binary candidates for both clouds:
smc_vis_binaries = pd.read_csv('../ysg_candidates/original_files_from_anna/smc_vis_bin.csv', comment='#')
smc_opt_binaries = pd.read_csv('../ysg_candidates/original_files_from_anna/smc_opt_bin.csv', comment='#') 
lmc_vis_binaries = pd.read_csv('../ysg_candidates/original_files_from_anna/lmc_vis_bin.csv', comment='#')
lmc_opt_binaries = pd.read_csv('../ysg_candidates/original_files_from_anna/lmc_opt_bin.csv', comment='#') 
# this is the file that contains the results of our variability analysis so far! 
# We load it here because it contains Anna's old T and L estimations and my estimations of the dominant period.
stats = pd.read_csv('../summary_results15.csv')
choose_surveys = pd.read_csv('choose_surveys_v3.csv')
# This is the results of fitting for the temperatures (comparing observed to synthetic photometry). Stores the best fits for every star
# temp_stats = pd.read_csv('ysg_temp_fitting_summary_v2.csv') # computations for U-redder, V-redder
temp_stats = pd.read_csv('ysg_temp_fitting_summary_v8_prefinal.csv') # computations for U-redder, B-redder, V-redder
# set serif font settings for plots
plt.rcParams['font.family'] = 'serif'

In [7]:
smcbasepath = './pysynphot_data/grid/bosz/r2000/m-0.75/'
lmcbasepath = './pysynphot_data/grid/bosz/r2000/m-0.25/'
wave=ascii.read('./pysynphot_data/grid/bosz/r2000/bosz2024_wave_r2000.txt', names=['wave'],data_start=0)

In [8]:
# Load in filter functions
col_names=['lam','flux']

tmass_j = ascii.read('./pysynphot_data/grid/bosz/2MASS_2MASS.J_v3.0596e-10_ab7.08741e-10_eff12350.dat', names=col_names, data_start=0)
tmass_h = ascii.read('./pysynphot_data/grid/bosz/2MASS_2MASS.H_v1.11064e-10_ab4.00078e-10_eff16620.dat', names=col_names, data_start=0)
tmass_k = ascii.read('./pysynphot_data/grid/bosz/2MASS_2MASS.Ks_v4.17999e-11_ab2.32482e-10_eff21590.dat', names=col_names, data_start=0)

mcps_U=ascii.read('./pysynphot_data/grid/bosz/Misc_MCPS.U_v4.08739e-9_ab8.23894e-9_eff3706.dat',names=col_names,data_start=0)
mcps_B=ascii.read('./pysynphot_data/grid/bosz/Misc_MCPS.B_v6.21086e-9_ab5.60999e-9_eff4394.dat', names=col_names,data_start=0)
mcps_V=ascii.read('./pysynphot_data/grid/bosz/Misc_MCPS.V_v3.64047e-9_ab3.63812e-9_eff5438.dat', names=col_names,data_start=0)
mcps_I=ascii.read('./pysynphot_data/grid/bosz/Misc_MCPS.I_v9.23651e-10_ab1.45234e-9_eff8568.dat', names=col_names,data_start=0)

swift_uvm1=ascii.read('./pysynphot_data/grid/bosz/Swift_UVOT.UVM2_trn_v4.66117e-9_ab2.15291e-8_eff2246.dat', names=col_names,data_start=0)
swift_uvw1=ascii.read('./pysynphot_data/grid/bosz/Swift_UVOT.UVW1_trn_v4.02204e-9_ab1.57569e-8_eff2715.dat', names=col_names,data_start=0)
swift_uvw2=ascii.read('./pysynphot_data/grid/bosz/Swift_UVOT.UVW2_trn_v5.37469e-9_2.59051e-8_eff2075.dat', names=col_names,data_start=0)  

# Additional bands:
apass_B=ascii.read('./pysynphot_data/grid/bosz/Generic_Johnson.B.dat', names=col_names,data_start=0)
apass_V=ascii.read('./pysynphot_data/grid/bosz/Generic_Johnson.V.dat', names=col_names,data_start=0)
apass_g=ascii.read('./pysynphot_data/grid/bosz/SLOAN_SDSS.g.dat', names=col_names,data_start=0)
apass_r=ascii.read('./pysynphot_data/grid/bosz/SLOAN_SDSS.r.dat', names=col_names,data_start=0)
apass_i=ascii.read('./pysynphot_data/grid/bosz/SLOAN_SDSS.i.dat', names=col_names,data_start=0)

smash_U=ascii.read('./pysynphot_data/grid/bosz/CTIO_DECam.u.dat', names=col_names,data_start=0)
smash_G=ascii.read('./pysynphot_data/grid/bosz/CTIO_DECam.g.dat', names=col_names,data_start=0)
smash_R=ascii.read('./pysynphot_data/grid/bosz/CTIO_DECam.r.dat', names=col_names,data_start=0)
smash_I=ascii.read('./pysynphot_data/grid/bosz/CTIO_DECam.i.dat', names=col_names,data_start=0)
smash_Z=ascii.read('./pysynphot_data/grid/bosz/CTIO_DECam.z.dat', names=col_names,data_start=0)


# Vega zero points for conversion to ergs/s/cm^2/Angstrom
tmass_j_zp = 3.0596e-10
tmass_h_zp = 1.11064e-10
tmass_k_zp = 4.17999e-11
mcps_U_zp = 4.08739e-9
mcps_B_zp = 6.21086e-9
mcps_V_zp = 3.64047e-9
mcps_I_zp = 9.23651e-10
swift_uvm1_zp = 4.66117e-9
swift_uvw1_zp = 4.02204e-9
swift_uvw2_zp = 5.37469e-9
# Additional bands:
apass_B_zp = 6.72553e-9
apass_V_zp = 3.636e-9
apass_g_zp = 4.92255e-9
apass_r_zp = 2.85425e-9
apass_i_zp = 1.94038e-9

smash_U_zp = 7.48186e-9
smash_G_zp = 4.70792e-9
smash_R_zp = 2.64299e-9
smash_I_zp = 1.78253e-9
smash_Z_zp = 1.29484e-9

apassband_wavelength = [4369.53,
                        5467.57,
                        4671.78,
                        6141.12,
                        7457.89]

smashband_wavelength = [3856.88,
                    4769.90,
                    6370.44,
                    7774.30,
                    9154.88]


# Band wavelengths for plotting
band_wavelengths = {
    'J': 12350.0,  
    'H': 16620.0,   
    'K': 21590.0,   
    'U': 3706.29,  
    'B': 4394.48,      
    'V': 5438.23,   
    'I': 8568.89,     
    'uvm2': 2246.56,  
    'uvw1': 2715.68,   
    'uvw2': 2075.69    
}

In [9]:
# Functions

def synth_flux(filter_name,model_lam,model_flux):
    '''
    Calculates the synthetic photometric flux for the input model with the
    input filter transmission file.

    Parameters:
    filter_name: The filter file containing the wavelength (Angstroms) 
                    and intensity (erg/s/cm^2/A)
    model_lam: The model wavelengths in Angstroms
    model_flux: The model flux in erg/s/cm^2/A

    Returns:
    flux: The synthetic flux in erg/s/cm^2/A
    '''
    filter_lam = filter_name.columns[0]
    filter_flux = filter_name.columns[1]
    
    f_lam=np.zeros(len(filter_lam))
    for i in range(len(f_lam)):
        # This interpolates the model flux along the same wavelengths as the
        # transmission curve. This needs to be done for the integration to work
        f_lam[i]=np.interp(filter_lam[i],model_lam,model_flux)
    
    # Multiply the (interpolated) model flux by the filter intensity, and integrate
    top = np.trapezoid(np.multiply(f_lam,filter_flux),x=filter_lam)
    # Integrate the filter intensity alone
    bottom = np.trapezoid(filter_flux,x=filter_lam)
    # Divide!
    flux = top/bottom
    
    return flux


def flux_to_mag(flux,zp):
    '''
    Converts a flux to a magnitude given the zero point.

    Parameters:
    flux: flux in erg/s/cm^2/A
    zp: Zero point in erg/s/cm^2/A

    Returns:
    mag: Magnitude
    '''

    mag = (-2.5)*np.log10(flux/zp)

    return mag

def synth_mag(filter_name,model_lam,model_flux,zp):
    '''
    Calculates the synthetic magnitude for the input model using the input
    filter transmission and a zero point.

    Parameters:
    filter_name: The filter file containing the wavelength (Angstroms) 
                    and intensity (erg/s/cm^2/A)
    model_lam: The model wavelengths in Angstroms
    model_flux: The model flux in erg/s/cm^2/A
    zp: Zero point in erg/s/cm^2/A

    Returns:
    mag: Magnitude
    '''

    flux = synth_flux(filter_name,model_lam,model_flux)
    mag = flux_to_mag(flux,zp)
    return mag

def rchi2_with_err(star_mags,star_err,model_mags):
    '''
    Returns the reduced chi^2, accounting for errors
    Parameters:
        star_mags: Observed magnitudes
        star_err: Uncertainty on the observed magnitudes
        model_mags: Model magnitudes
    Returns:
        rchi2: Reduced chi^2 value
    '''
    N = len(star_mags)
    z = (star_mags-model_mags)/star_err
    rchi2 = np.sum(z**2)/(N-1)
    return rchi2

def Cardelli_redden(wave,flux,Av=0,Rv=3.1):
    '''Will redden an input spectrum based on the Cardelli law:
    
    Parameters:
    wave (angstrom): wavelengths of input spectrum in Angstrom
    flux (erg/s/cm/Ang): flux of spectrum; scaled version of this flux are fine.
    Av:
    Rv:
    
    Returns: a new flux array in same units as input
    '''

    wave_micon=wave*1.0e-4
    wave_inverse=1.0/wave_micon
    
    A_lambda=np.zeros(len(wave))
    ax=np.zeros(len(wave))
    bx=np.zeros(len(wave))
    
    for i in range(len(wave)):
        if((wave_inverse[i] > 0.3) and (wave_inverse[i] < 1.1)):
            ax[i]=0.574*wave_inverse[i]**1.61
            bx[i]=-0.527*wave_inverse[i]**1.61
        if((wave_inverse[i] > 1.1) and (wave_inverse[i] < 3.3)):
            y=wave_inverse[i] - 1.82
            ax[i]=1+ 0.17699*y - 0.50447*y**2. - 0.02427*y**3.+0.72085*y**4.+0.01979*y**5.-0.77530*y**6.+0.32999*y**7.
            bx[i]=1.41338*y+2.28305*y**2.+1.07233*y**3.-5.38434*y**4.-0.62251*y**5.+5.30260*y**6.-2.09002*y**7.
        if((wave_inverse[i] > 3.3) and (wave_inverse[i] < 8.0)):
            if(wave_inverse[i] > 5.9):
                Fa=-0.04473*(wave_inverse[i] - 5.9)**2.-0.009779*(wave_inverse[i]-5.9)**3.
                Fb=0.2130*(wave_inverse[i] - 5.9)**2. +0.1207*(wave_inverse[i] -5.9)**3.
            else:
                Fa = 0.0
                Fb = 0.0

            ax[i]=1.752 - 0.316*wave_inverse[i]-0.104/((wave_inverse[i]-4.67)**2.+0.341) +Fa
            bx[i]=-3.090+1.825*wave_inverse[i]+1.206/((wave_inverse[i] - 4.62)**2.+0.263) +Fb

    A_lambda=(ax+bx/Rv)*Av
    #print(A_lambda)
    NewSpec=flux*10.**(-0.4*A_lambda)
    results = NewSpec
    return results

def compute_synth_photometry(ex_wave, ex_flux):
    jmag=synth_mag(tmass_j,ex_wave['wave'],ex_flux['flux'],tmass_j_zp)
    hmag=synth_mag(tmass_h,ex_wave['wave'],ex_flux['flux'],tmass_h_zp)
    kmag=synth_mag(tmass_k,ex_wave['wave'],ex_flux['flux'],tmass_k_zp)
    Umag=synth_mag(mcps_U,ex_wave['wave'],ex_flux['flux'],mcps_U_zp)
    Bmag=synth_mag(mcps_B,ex_wave['wave'],ex_flux['flux'],mcps_B_zp)
    Vmag=synth_mag(mcps_V,ex_wave['wave'],ex_flux['flux'],mcps_V_zp)
    Imag=synth_mag(mcps_I,ex_wave['wave'],ex_flux['flux'],mcps_I_zp)
    uvm2_mag=synth_mag(swift_uvm1,ex_wave['wave'],ex_flux['flux'],swift_uvm1_zp)
    uvw1_mag=synth_mag(swift_uvw1,ex_wave['wave'],ex_flux['flux'],swift_uvw1_zp)
    uvw2_mag=synth_mag(swift_uvw2,ex_wave['wave'],ex_flux['flux'],swift_uvw2_zp)
    model_mags=[jmag,hmag,kmag,Umag,Bmag,Vmag,Imag,uvm2_mag,uvw1_mag,uvw2_mag]

    #convert them to flux:
    jflux = tmass_j_zp * 10**(-0.4 * jmag)
    hflux = tmass_h_zp * 10**(-0.4 * hmag)
    kflux = tmass_k_zp * 10**(-0.4 * kmag)
    Uflux = mcps_U_zp * 10**(-0.4 * Umag)
    Bflux = mcps_B_zp * 10**(-0.4 * Bmag)
    Vflux = mcps_V_zp * 10**(-0.4 * Vmag)
    Iflux = mcps_I_zp * 10**(-0.4 * Imag)
    uvm2_flux = swift_uvm1_zp * 10**(-0.4 * uvm2_mag)
    uvw1_flux = swift_uvw1_zp * 10**(-0.4 * uvw1_mag)
    uvw2_flux = swift_uvw2_zp * 10**(-0.4 * uvw2_mag)
    model_fluxes=[jflux,hflux,kflux,Uflux,Bflux,Vflux,Iflux,uvm2_flux,uvw1_flux,uvw2_flux]

    # dictionary with band names as keys
    synth_phot = {
        'J': {'mag': jmag, 'flux': jflux},
        'H': {'mag': hmag, 'flux': hflux},
        'K': {'mag': kmag, 'flux': kflux},
        'U': {'mag': Umag, 'flux': Uflux},
        'B': {'mag': Bmag, 'flux': Bflux},
        'V': {'mag': Vmag, 'flux': Vflux},
        'I': {'mag': Imag, 'flux': Iflux},
        'uvm2': {'mag': uvm2_mag, 'flux': uvm2_flux},
        'uvw1': {'mag': uvw1_mag, 'flux': uvw1_flux},
        'uvw2': {'mag': uvw2_mag, 'flux': uvw2_flux}
    }
    return synth_phot

In [10]:
def smash_apass(index):
    RA = coords['RA'].iloc[index]
    dec = coords['DEC'].iloc[index]
    
    if index < 377:
        row = df_smc[(df_smc['ra'] == RA) & (df_smc['dec'] == dec)]
    else:
        row = df_lmc[(df_lmc['ra'] == RA) & (df_lmc['dec'] == dec)] 

    smashmags = []
    smashmag_errs = []
    smashflux_jy = []
    smashflux_err_jy = []

    # Bands and their AB ZEROPOINTS in erg/s/cm^2/Angstrom (in UGRIZ order)
    smashband_zeropoints = {'U': 7.48186e-9,
                       'G': 4.70792e-9,
                       'R': 2.64299e-9,
                       'I': 1.78253e-9,
                       'Z': 1.29484e-9}

    # Effective wavelengths (in Angstroms) (in UGRIZ order)
    smashband_wavelength = [3856.88,
                       4769.90,
                       6370.44,
                       7774.30,
                       9154.88]
    
    smash_band_names = ['U', 'G', 'R', 'I', 'Z']

    # do this for UGRIZ mags
    for i, band in enumerate(smash_band_names):
        mag_col = f'{band}smashmag'
        err_col = f'e_{band}smash'
        
        if len(row) > 0 and not pd.isna(row[mag_col].values[0]):
            magvalue = row[mag_col].values[0]
            magerr = row[err_col].values[0]
            
            # Convert magnitude to flux density
            flux_val = smashband_zeropoints[band] * 10**(-0.4 * magvalue)
            flux_err_val = flux_val * 0.921 * magerr
            
            smashmags.append(magvalue)
            smashmag_errs.append(magerr)
            smashflux_jy.append(flux_val)
            smashflux_err_jy.append(flux_err_val)
        else:
            smashmags.append(np.nan)
            smashmag_errs.append(np.nan)
            smashflux_jy.append(np.nan)
            smashflux_err_jy.append(np.nan)
    
    apassmags = []
    apassmag_errs = []
    apassflux_jy = []
    apassflux_err_jy = []

    apassband_zeropoints = {'B': 6.72553e-9, # vega, B
                            'V': 3.636e-9, # vega, V
                            'G': 4.92255e-9, # ab, g
                            'R': 2.85425e-9, # ab, r
                            'I': 1.94038e-9 # ab, i
                            } 
    
    apassband_wavelength = [4369.53,
                            5467.57,
                            4671.78,
                            6141.12,
                            7457.89]
    
    apass_band_names = ['B', 'V', 'G', 'R', 'I']
    
    for i, band in enumerate(apass_band_names):
        mag_col = f'{band}mag'
        # err_col = f'e_{band}apass'
        apassfile = pd.read_csv('UCAC4_match.csv')
        row = apassfile[(apassfile['RA'] == RA) & (apassfile['DEC'] == dec)]
        
        if not pd.isna(row[mag_col].values[0]):
            magvalue = row[mag_col].values[0]
            # magerr = row[err_col].values[0]
            
            # Convert magnitude to flux density
            flux_val = apassband_zeropoints[band] * 10**(-0.4 * magvalue)
            # flux_err_val = flux_val * 0.921 * magerr
            
            apassmags.append(magvalue)
            # apassmag_errs.append(magerr)
            apassflux_jy.append(flux_val)
            # apassflux_err_jy.append(flux_err_val)
        else:
            apassmags.append(np.nan)
            # apassmag_errs.append(np.nan)
            apassflux_jy.append(np.nan)
            # apassflux_err_jy.append(np.nan)
    

    return smashband_wavelength, smashflux_jy, smashflux_err_jy, smashmags, smashmag_errs, smash_band_names, apassband_wavelength, apassflux_jy, apassmags, apass_band_names #apassmag_errs, apassflux_err_jy,

smash_apass(0)

FileNotFoundError: [Errno 2] No such file or directory: 'UCAC4_match.csv'

In [11]:
def histo(index):
    RA = coords['RA'].iloc[index]
    dec = coords['DEC'].iloc[index]
    binwidth = 250
    final_teff_U_redder = temp_stats['teff_mean_U_redder'].iloc[index]
    final_teff_V_redder = temp_stats['teff_mean_V_redder'].iloc[index]
    output_filename_U_redder = f'temp_fitting/{RA}_{dec}_U_redder.parquet'
    output_filename_V_redder = f'temp_fitting/{RA}_{dec}_V_redder.parquet'
    # read parquet file:
    model_fitting_U_redder = pd.read_parquet(output_filename_U_redder)
    model_fitting_V_redder = pd.read_parquet(output_filename_V_redder)
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 12))
    
    # Create bins that properly include the max value
    bins_U_redder = range(min(model_fitting_U_redder['teff']) - (binwidth//2), 
                      max(model_fitting_U_redder['teff']) + binwidth//2 + 1, binwidth)
    bins_V_redder = range(min(model_fitting_V_redder['teff']) - binwidth//2, 
                     max(model_fitting_V_redder['teff']) + binwidth//2 + 1, binwidth)
    bins_U_redder_logg = int((max(model_fitting_U_redder['logg']) - min(model_fitting_U_redder['logg'])) * 2.0)
    bins_V_redder_logg = int((max(model_fitting_V_redder['logg']) - min(model_fitting_V_redder['logg'])) * 2.0)
    # Ensure minimum number of bins
    bins_U_redder_logg = max(bins_U_redder_logg, 1)
    bins_V_redder_logg = max(bins_V_redder_logg, 1)  
    
    # Top row - 'full' plots in blue
    axes[0, 0].hist(model_fitting_U_redder['teff'], bins=bins_U_redder, color='blue', alpha=0.7, edgecolor='black')
    axes[0, 0].annotate(f'Average Teff = {final_teff_U_redder:.0f}K \nAverage $\\chi^2$ = {temp_stats.loc[index, "chi2_U_redder_mean"]:.3f}', xy=(0.5, 0.9), xycoords='axes fraction', fontsize=12)
    axes[0, 0].set_xlabel('$T_{eff}$ (K)', fontsize=16)
    axes[0, 0].set_ylabel('Counts')
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].hist(model_fitting_U_redder['logL'], bins=len(bins_U_redder), color='blue', alpha=0.7, edgecolor='black')
    axes[0, 1].set_xlabel('log($L/L_{\\odot})$', fontsize=16)
    axes[0, 1].set_ylabel('Counts')
    axes[0, 1].grid(True, alpha=0.3)

    # Define custom bin edges for A_v and log(g)
    av_bins = np.arange(-0.05, 1.15, 0.1)  # Centers on 0, 0.1, 0.2, ..., 1.0
    logg_bins = np.arange(-0.25, 3.75, 0.5)  # Centers on 0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0

    axes[0, 2].hist(model_fitting_U_redder['av'], bins=av_bins, color='blue', alpha=0.7, edgecolor='black')
    axes[0, 2].set_xlabel('$A_v$', fontsize=16)
    axes[0, 2].set_ylabel('Counts')
    axes[0, 2].grid(True, alpha=0.3)

    axes[0, 3].hist(model_fitting_U_redder['logg'], bins=logg_bins, color='blue', alpha=0.7, edgecolor='black')
    axes[0, 3].set_xlabel('log(g)', fontsize=16)
    axes[0, 3].set_ylabel('Counts')
    axes[0, 3].grid(True, alpha=0.3)
    
    # Bottom row - 'cut' plots in green
    axes[1, 0].hist(model_fitting_V_redder['teff'], bins=bins_V_redder, color='green', alpha=0.7, edgecolor='black')
    axes[1, 0].annotate(f'Average Teff = {final_teff_V_redder:.0f}K \nAverage $\\chi^2$ = {temp_stats.loc[index, "chi2_V_redder_mean"]:.3f}', xy=(0.5, 0.9), xycoords='axes fraction', fontsize=12)
    axes[1, 0].set_xlabel('$T_{eff}$ (K)', fontsize=16)
    axes[1, 0].set_ylabel('Counts')
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].hist(model_fitting_V_redder['logL'], bins=len(bins_V_redder), color='green', alpha=0.7, edgecolor='black')
    axes[1, 1].set_xlabel('log($L/L_{\\odot}$)', fontsize=16)
    axes[1, 1].set_ylabel('Counts')
    axes[1, 1].grid(True, alpha=0.3)

    axes[1, 2].hist(model_fitting_V_redder['av'], bins=av_bins, color='green', alpha=0.7, edgecolor='black')
    axes[1, 2].set_xlabel('$A_v$', fontsize=16)
    axes[1, 2].set_ylabel('Counts')
    axes[1, 2].grid(True, alpha=0.3)

    axes[1, 3].hist(model_fitting_V_redder['logg'], bins=logg_bins, color='green', alpha=0.7, edgecolor='black')
    axes[1, 3].set_xlabel('log(g)', fontsize=16)
    axes[1, 3].set_ylabel('Counts')
    axes[1, 3].grid(True, alpha=0.3)
    
    # Add row labels
    fig.text(-0.005, 0.75, 'Fitting to UBVIJHK', rotation=90, fontsize=16, va='center', ha='center')
    fig.text(-0.005, 0.25, 'Fitting to VIJHK', rotation=90, fontsize=16, va='center', ha='center')
    fig.text(0.4,-0.02, f'{RA}, {dec}', fontsize=22)
    
    plt.tight_layout()
    plt.show()

# comment this in if you downloaded the parquet files!
# histo(0)

In [12]:
def fit_models_to_star_flux(star_idx): 
    """
    Fits the best model spectra (the median) to the observed SED of a given star index,
    comparing models based on median parameters from temp_stats.
    Optionally plots the observed SED and best-fit models.
    """  
    computed_models = pd.read_csv('synth_phot_all_models_allphot_gordon.csv')

    ######### grab the observed SED for the star
    obs = observed_sed(star_idx, show=False)
    obs_wavelengths, obs_fluxes, obs_flux_errors, obs_mags, obs_mag_errors, obs_band_names = obs

    smash_wavelengths, smash_fluxes, smash_flux_errors, smash_mags, smash_mag_errors, smash_band_names = smash_apass(star_idx)[:6]
    apass_wavelengths, apass_fluxes, apass_mags, apass_band_names = smash_apass(star_idx)[6:]

    
    available_bands = list(obs_band_names)  # Convert to list to ensure .index() method works
    additional_bands = list(smash_band_names) + list(apass_band_names)
    
    band_to_filter = {
        'J': tmass_j, 'H': tmass_h, 'K': tmass_k,
        'U': mcps_U, 'B': mcps_B, 'V': mcps_V, 'I': mcps_I,
        'uvm2': swift_uvm1, 'uvw1': swift_uvw1, 'uvw2': swift_uvw2
    }     

    smash_bands_to_filter = {
        'U': smash_U, 'G': smash_G, 'R': smash_R, 'I': smash_I, 'Z': smash_Z}
    apass_bands_to_filter = {
        'B': apass_B, 'V': apass_V, 'G': apass_g, 'R': apass_r, 'I': apass_i
    }

    # additional_band_to_filter = {
    #     'U': mcps_U, 'G': None, 'R': None, '

    ########## determine the best fitting model to test based on medians in ysg_temp_fitting_summary_v3.csv
    # Best fitting U-redder parameters:
    median_teff_U_redder = temp_stats['teff_median_U_redder'][star_idx]
    median_logg_U_redder = temp_stats['logg_median_U_redder'][star_idx]
    median_av_U_redder_raw = temp_stats['av_median_U_redder'][star_idx]  # Keep raw value for display
    median_av_U_redder = round(np.round(median_av_U_redder_raw / 0.05) * 0.05, 2)  # Round to nearest 0.05 for model lookup
    median_metallicity = -0.75 if star_idx < 377 else -0.25 # SMC if index < 377, else LMC
    
    # Best fitting B-redder parameters:
    median_teff_B_redder = temp_stats['teff_median_B_redder'][star_idx]
    median_logg_B_redder = temp_stats['logg_median_B_redder'][star_idx]
    median_av_B_redder_raw = temp_stats['av_median_B_redder'][star_idx]  # Keep raw value for display
    median_av_B_redder = round(np.round(median_av_B_redder_raw / 0.05) * 0.05, 2)  # Round to nearest 0.05 for model lookup

    #Best fitting V-redder parameters:
    median_teff_V_redder = temp_stats['teff_median_V_redder'][star_idx]
    median_logg_V_redder = temp_stats['logg_median_V_redder'][star_idx]
    median_av_V_redder_raw = temp_stats['av_median_V_redder'][star_idx]  # Keep raw value for display
    median_av_V_redder = round(np.round(median_av_V_redder_raw / 0.05) * 0.05, 2)  # Round to nearest 0.05 for model lookup

    # print(f"Star {star_idx} Av rounding:")
    # print(f"  U_redder: {median_av_U_redder_raw:.3f} to {median_av_U_redder:.2f}")
    # print(f"  B_redder: {median_av_B_redder_raw:.3f} to {median_av_B_redder:.2f}")
    # print(f"  V_redder: {median_av_V_redder_raw:.3f} to {median_av_V_redder:.2f}")

    # # try closest to the mean teff for plotting
    # mean_teff_U_redder = temp_stats['teff_mean_U_redder'][star_idx]
    # mean_logg_U_redder = temp_stats['logg_mean_U_redder'][star_idx]
    # mean_av_U_redder = temp_stats['av_mean_U_redder'][star_idx]

    # mean_teff_B_redder = temp_stats['teff_mean_B_redder'][star_idx]
    # mean_logg_B_redder = temp_stats['logg_mean_B_redder'][star_idx]
    # mean_av_B_redder = temp_stats['av_mean_B_redder'][star_idx] 

    # mean_teff_V_redder = temp_stats['teff_mean_V_redder'][star_idx]
    # mean_logg_V_redder = temp_stats['logg_mean_V_redder'][star_idx]
    # mean_av_V_redder = temp_stats['av_mean_V_redder'][star_idx]


    # Filter models based on median parameters - separate the two model sets properly
    # Use rounded Av values for exact model lookup
    models_U_redder = computed_models[
        (computed_models['teff'] == median_teff_U_redder) &
        (computed_models['logg'] == median_logg_U_redder) &
        (computed_models['av'] == median_av_U_redder) &
        (computed_models['metallicity'] == median_metallicity)
    ]
    
    models_B_redder = computed_models[
        (computed_models['teff'] == median_teff_B_redder) &
        (computed_models['logg'] == median_logg_B_redder) &
        (computed_models['av'] == median_av_B_redder) &
        (computed_models['metallicity'] == median_metallicity)
    ]

    models_V_redder = computed_models[
        (computed_models['teff'] == median_teff_V_redder) &
        (computed_models['logg'] == median_logg_V_redder) &
        (computed_models['av'] == median_av_V_redder) &
        (computed_models['metallicity'] == median_metallicity)
    ]
    
    plot_data = []
    
    # Process all models (full, B-redder and V-redder)
    for model_type, models in [('V_redder', models_V_redder), ('B_redder', models_B_redder), ('U_redder', models_U_redder)]:
        if len(models) == 0:
            print(f"No models found for {model_type}")
            print(f"  Parameters: Teff={median_teff_V_redder if model_type=='V_redder' else (median_teff_B_redder if model_type=='B_redder' else median_teff_U_redder)}, logg={median_logg_V_redder if model_type=='V_redder' else (median_logg_B_redder if model_type=='B_redder' else median_logg_U_redder)}, Av={median_av_V_redder if model_type=='V_redder' else (median_av_B_redder if model_type=='B_redder' else median_av_U_redder)}, metallicity={median_metallicity}")
            continue
            
        # take the first (should be only) model
        model = models.iloc[0]
        
        # load model spectrum
        model_spectrum = ascii.read(model['model'], names=['flux','cont'], data_start=0)
        model_flux = Cardelli_redden(wave['wave'], model_spectrum['flux'], Av=model['av'])
        
        # calculate synthetic fluxes for available bands using the model
        model_fluxes = np.array([synth_flux(band_to_filter[band], wave['wave'], model_flux) for band in available_bands])
        smash_model_fluxes = np.array([synth_flux(smash_bands_to_filter[band], wave['wave'], model_flux) for band in smash_band_names])
        apass_model_fluxes = np.array([synth_flux(apass_bands_to_filter[band], wave['wave'], model_flux) for band in apass_band_names])
        
        # Determine reference band for flux scaling (prefer K, then H)
        if 'K' in available_bands:
            ref_band = 'K'
        else:
            print('No K band found in observed bands.')
            return
        
        ref_idx = available_bands.index(ref_band)
        
        # Calculate scaling factor using reference band
        flux_scale = obs_fluxes[ref_idx] / model_fluxes[ref_idx]
        
        # Scale model fluxes and spectrum
        scaled_model_fluxes = model_fluxes * flux_scale
        scaled_smash_model_fluxes = smash_model_fluxes * flux_scale
        scaled_apass_model_fluxes = apass_model_fluxes * flux_scale
        scaled_model_spectrum = model_flux * flux_scale
        
        # Calculate luminosity from scaling
        luminosity = model['lum_unscaled'] * flux_scale
        logL = np.log10(luminosity / 3.826e33)  # solar lum in erg/s
        
        # Get wavelengths for plotting
        plot_wavelengths = [band_wavelengths[band] for band in available_bands]
        plot_smash_wavelengths = [smashband_wavelength[i] for i, band in enumerate(smash_band_names)]
        plot_apass_wavelengths = [apassband_wavelength[i] for i, band in enumerate(apass_band_names)]
        
        # Store data for plotting
        plot_data.append({
            'model_type': model_type,
            'model': model,
            'scaled_model_fluxes': scaled_model_fluxes,
            'scaled_model_spectrum': scaled_model_spectrum,
            'plot_wavelengths': plot_wavelengths,
            'scaled_smash_model_fluxes': scaled_smash_model_fluxes,
            'scaled_apass_model_fluxes': scaled_apass_model_fluxes,
            'plot_smash_wavelengths': plot_smash_wavelengths,
            'plot_apass_wavelengths': plot_apass_wavelengths,
            'obs_fluxes': obs_fluxes,
            'available_bands': available_bands,
            'logL': logL,
            'luminosity': luminosity
        })


    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    RA = coords['RA'].iloc[star_idx]
    dec = coords['DEC'].iloc[star_idx]
    
    # Plot observed data points 
    ax.errorbar(obs_wavelengths, obs_fluxes, yerr=obs_flux_errors,
                ms=10, fmt='*', mec='C0', mfc='lightblue', ecolor='C0', label='Observed (UVOT/MCPS/2MASS)', zorder=10)
    
    # Plot V-redder synthetic photometry
    data_V_redder = plot_data[0]
    ax.plot(data_V_redder['plot_wavelengths'], data_V_redder['scaled_model_fluxes'],
            'o', ms=10, mec='k', mfc='red', alpha=0.9, label='V-redder Synth Phot, Teff={:.0f}K'.format(data_V_redder['model']['teff']), zorder=8)
    ax.plot(data_V_redder['plot_smash_wavelengths'], data_V_redder['scaled_smash_model_fluxes'],
            's', ms=8, mec='k', mfc='red', alpha=0.9, label='SMASH Model Points', zorder=8)
    ax.plot(data_V_redder['plot_apass_wavelengths'], data_V_redder['scaled_apass_model_fluxes'],
            'D', ms=8, mec='k', mfc='red', alpha=0.9, label='APASS Model Points', zorder=8)
    
    # Plot B-redder synthetic photometry
    data_B_redder = plot_data[1]
    ax.plot(data_B_redder['plot_wavelengths'], data_B_redder['scaled_model_fluxes'],
            'o', ms=10, mec='k', mfc='yellow', alpha=0.9, label='B-redder Synth Phot, Teff={:.0f}K'.format(data_B_redder['model']['teff']), zorder=8)
    ax.plot(data_B_redder['plot_smash_wavelengths'], data_B_redder['scaled_smash_model_fluxes'],
            's', ms=8, mec='k', mfc='yellow', alpha=0.9, label='SMASH Model Points', zorder=8)
    ax.plot(data_B_redder['plot_apass_wavelengths'], data_B_redder['scaled_apass_model_fluxes'],
            'D', ms=8, mec='k', mfc='yellow', alpha=0.9, label='APASS Model Points', zorder=8)
    
    # Plot U-redder synthetic photometry
    data_U_redder = plot_data[2] 
    ax.plot(data_U_redder['plot_wavelengths'], data_U_redder['scaled_model_fluxes'],
            'o', ms=10, mec='k', mfc='blue', alpha=0.9, label='U-redder Synth Phot, Teff={:.0f}K'.format(data_U_redder['model']['teff']), zorder=8)
    ax.plot(data_U_redder['plot_smash_wavelengths'], data_U_redder['scaled_smash_model_fluxes'],
            's', ms=8, mec='k', mfc='blue', alpha=0.9, label='SMASH Model Points', zorder=8)
    ax.plot(data_U_redder['plot_apass_wavelengths'], data_U_redder['scaled_apass_model_fluxes'],
            'D', ms=8, mec='k', mfc='blue', alpha=0.9, label='APASS Model Points', zorder=8)
    
    # band labels for observed points
    for i, (wl, flux, band) in enumerate(zip(obs_wavelengths, obs_fluxes, obs_band_names)):
        ax.annotate(band, (wl, flux), xytext=(5, 5), textcoords='offset points', 
                    fontsize=8, alpha=0.7)
    
    # model spectra in background
    ax.plot(wave['wave'], data_V_redder['scaled_model_spectrum'],
            '-', color='red', alpha=0.6, label='V-redder Spectrum', zorder=5)
    ax.plot(wave['wave'], data_B_redder['scaled_model_spectrum'],
            '-', color='yellow', alpha=0.6, label='B-redder Model Spectrum', zorder=5)
    ax.plot(wave['wave'], data_U_redder['scaled_model_spectrum'],
            '-', color='blue', alpha=0.6, label='U-redder Model Spectrum', zorder=5)
    
    ax.grid(True, alpha=0.3)

    # inset axes in the lower right corner
    axins = inset_axes(ax, width="35%", height="35%", loc='upper right')

    # Plot the zoom to U and B band region
    axins.errorbar(obs_wavelengths, obs_fluxes, yerr=obs_flux_errors,
                ms=10, fmt='*', mec='C0', mfc='lightblue', ecolor='C0', zorder=10)
    axins.plot(data_V_redder['plot_wavelengths'], data_V_redder['scaled_model_fluxes'],
            'o', ms=8, mec='k', mfc='red', alpha=0.7, zorder=8)
    axins.plot(data_V_redder['plot_smash_wavelengths'], data_V_redder['scaled_smash_model_fluxes'],
            's', ms=6, mec='k', mfc='red', alpha=0.7, zorder=8)
    axins.plot(data_V_redder['plot_apass_wavelengths'], data_V_redder['scaled_apass_model_fluxes'],
            'D', ms=6, mec='k', mfc='red', alpha=0.7, zorder=8)
    axins.plot(data_B_redder['plot_wavelengths'], data_B_redder['scaled_model_fluxes'],
        'o', ms=8, mec='k', mfc='yellow', alpha=0.7, zorder=8)
    axins.plot(data_B_redder['plot_smash_wavelengths'], data_B_redder['scaled_smash_model_fluxes'],
            's', ms=6, mec='k', mfc='yellow', alpha=0.7, zorder=8)
    axins.plot(data_B_redder['plot_apass_wavelengths'], data_B_redder['scaled_apass_model_fluxes'],
            'D', ms=6, mec='k', mfc='yellow', alpha=0.7, zorder=8)
    axins.plot(data_U_redder['plot_wavelengths'], data_U_redder['scaled_model_fluxes'],
        'o', ms=8, mec='k', mfc='blue', alpha=0.7, zorder=8)
    axins.plot(data_U_redder['plot_smash_wavelengths'], data_U_redder['scaled_smash_model_fluxes'],
            's', ms=6, mec='k', mfc='blue', alpha=0.7, zorder=8)
    axins.plot(data_U_redder['plot_apass_wavelengths'], data_U_redder['scaled_apass_model_fluxes'],
            'D', ms=6, mec='k', mfc='blue', alpha=0.7, zorder=8)
    axins.plot(wave['wave'], data_V_redder['scaled_model_spectrum'],
            '-', color='red', alpha=0.6, label='V-redder Model Spectrum', zorder=5)
    axins.plot(wave['wave'], data_B_redder['scaled_model_spectrum'],
            '-', color='yellow', alpha=0.6, label='B-redder Model Spectrum', zorder=5)
    axins.plot(wave['wave'], data_U_redder['scaled_model_spectrum'],
            '-', color='blue', alpha=0.6, label='U-redder Model Spectrum', zorder=5)

    # Set zoom limits around U band
    axins.set_xlim(3500, 4800) 
    # Get U band flux values specifically
    u_idx_V_redder = data_V_redder['available_bands'].index('U')
    u_idx_B_redder = data_B_redder['available_bands'].index('U')
    u_idx_U_redder = data_U_redder['available_bands'].index('U')
    u_idx_obs = np.where(obs_band_names == 'U')[0][0]
    
    u_fluxes = [
        data_V_redder['scaled_model_fluxes'][u_idx_V_redder],
        data_B_redder['scaled_model_fluxes'][u_idx_B_redder],
        data_U_redder['scaled_model_fluxes'][u_idx_U_redder],
        obs_fluxes[u_idx_obs]
    ]
    
    minimum = np.min(u_fluxes) * 0.5  
    maximum = np.max(u_fluxes) * 2.0  
    axins.set_ylim(minimum, maximum)  # Set appropriate flux range
    axins.set_yscale('log')
    axins.set_xscale('log')
    # Remove all tick labels and ticks - must be done AFTER setting scales
    axins.set_xticks([])
    axins.set_yticks([])
    axins.tick_params(axis='both', which='both', bottom=False, top=False, 
                        left=False, right=False, labelbottom=False, labeltop=False,
                        labelleft=False, labelright=False, length=0)
    # Add title to inset
    # axins.set_title('U and B zoom-in', fontsize=12, pad=5) 
    axins.set_xlabel('U and B zoom-in', fontsize=12, labelpad=5) 
    
    # Plot additional models within +/-500K range
    for model_type, main_data, color in [('V_redder', data_V_redder, 'gold'), ('B_redder', data_B_redder, 'violet'), ('U_redder', data_U_redder, 'cyan')]:
        best_teff = main_data['model']['teff']
        best_logg = main_data['model']['logg']
        best_av = main_data['model']['av']
        best_metallicity = main_data['model']['metallicity']
        
        # Determine temperature increment based on the model grid: 250K below 12000K, 500K at/above 12000K
        if best_teff < 12000:
            temp_increment = 250
        else:
            temp_increment = 500
        
        # Find temperature variants: one below and one above the best fit
        temp_low = best_teff - temp_increment
        temp_high = best_teff + temp_increment
        
        # Determine allowed logg values based on temperature
        # At 12000K: logg can be 2.5 or 3.0
        # At 12500K and above: only logg = 3.0 is possible
        allowed_logg = [best_logg]  # Always include the best fit logg
        
        if temp_low == 12000 or temp_high == 12000:
            # At 12000K, allow both 2.5 and 3.0
            if 2.5 not in allowed_logg:
                allowed_logg.append(2.5)
            if 3.0 not in allowed_logg:
                allowed_logg.append(3.0)
        elif temp_high >= 12500:
            # At 12500K and above, only logg=3.0
            allowed_logg = [3.0]
        
        temp_models = computed_models[
            ((computed_models['teff'] == temp_low) | (computed_models['teff'] == temp_high)) &
            (computed_models['logg'].isin(allowed_logg)) &  # Surface gravity in allowed range
            (computed_models['av'] == best_av) &       # Same extinction
            (computed_models['metallicity'] == best_metallicity)
        ]
        # print(f"Found {len(temp_models)} temperature variant models within {temp_increment}K:")
        # print(f"Available temperatures: {temp_models['teff'].unique()}")
        
    #     # Limit to the two closest temperature variants
    #     temp_models = temp_models.head(2)
        
    #     for r, temp_model in temp_models.iterrows():
    #         try:
    #             # Load and process temperature variant model
    #             temp_spectrum = ascii.read(temp_model['model'], names=['flux','cont'], data_start=0)
    #             temp_flux = Cardelli_redden(wave['wave'], temp_spectrum['flux'], Av=temp_model['av'])
                
    #             # Calculate fluxes for common bands using the same mapping
    #             temp_model_fluxes = np.array([synth_flux(band_to_filter[band], wave['wave'], temp_flux) for band in available_bands])
                
    #             # Scale using same reference band as main model (ref_band and ref_idx already determined)
    #             main_ref_idx = main_data['available_bands'].index(ref_band)
    #             obs_ref_flux = main_data['obs_fluxes'][main_ref_idx]
    #             temp_flux_scale = obs_ref_flux / temp_model_fluxes[ref_idx]
                
    #             # Plot the scaled spectrum with very light alpha
    #             temp_scaled_spectrum = temp_flux * temp_flux_scale
    #             ax.plot(wave['wave'], temp_scaled_spectrum,
    #                     '-', color=color, alpha=0.3, linewidth=0.5)
    #             axins.plot(wave['wave'], temp_scaled_spectrum,
    #             '-', color=color, alpha=0.3)
                
    #         except Exception as e:
    #             print(f"Warning: could not process temperature variant model {temp_model['teff']}K: {e}")
    #             continue
    # # Just replot the last one for 1 label to appear in the legend
    # ax.plot(wave['wave'], temp_scaled_spectrum,
    #                     '-', color=color, alpha=0.3, linewidth=0.5, label = 'Closest spectra')


    # overplotting usmash mags
    RA = coords['RA'].iloc[star_idx]
    dec = coords['DEC'].iloc[star_idx]


    ######### NEW PHOTOMETRY FROM SMASH AND APASS

    ax.errorbar(smash_wavelengths, smash_fluxes, yerr=smash_flux_errors,
                ms=10, fmt='*', mec='palevioletred', mfc='lightpink', ecolor='k', label='SMASH', zorder=10)
    axins.errorbar(smash_wavelengths, smash_fluxes, yerr=smash_flux_errors,
                ms=10, fmt='*', mec='palevioletred', mfc='lightpink', ecolor='k', label='SMASH', zorder=10)
    for i, (wl, flux, band) in enumerate(zip(smash_wavelengths, smash_fluxes, smash_band_names)):
        ax.annotate(band, (wl, flux), xytext=(-2, -20), textcoords='offset points', 
                    fontsize=8, color = 'red', alpha=0.7)
        axins.annotate(band, (wl, flux), xytext=(-2, -20), textcoords='offset points', 
                    fontsize=8, color = 'red', alpha=0.7)

    ax.errorbar(apass_wavelengths, apass_fluxes,
                ms=10, fmt='*', mec='darkgreen', mfc='lime', ecolor='k', label='APASS', zorder=10)
    axins.errorbar(apass_wavelengths, apass_fluxes,
                ms=10, fmt='*', mec='darkgreen', mfc='lime', ecolor='k', zorder=10)
    for i, (wl, flux, band) in enumerate(zip(apass_wavelengths, apass_fluxes, apass_band_names)):
        ax.annotate(band, (wl, flux), xytext=(-2, -15), textcoords='offset points', 
                    fontsize=8, color = 'lime', alpha=0.7)
        axins.annotate(band, (wl, flux), xytext=(-2, -20), textcoords='offset points', 
                    fontsize=8, color = 'lime', alpha=0.7)
    
    # Check if star is in binary candidate files
    tolerance = 1e-5  # Small tolerance for coordinate matching
    in_optsmc = ((np.abs(smc_opt_binaries['ra'] - RA) < tolerance) & 
                (np.abs(smc_opt_binaries['dec'] - dec) < tolerance)).any()
    in_optlmc = ((np.abs(lmc_opt_binaries['ra'] - RA) < tolerance) & 
                (np.abs(lmc_opt_binaries['dec'] - dec) < tolerance)).any()
    in_vissmc = ((np.abs(smc_vis_binaries['ra'] - RA) < tolerance) & 
                (np.abs(smc_vis_binaries['dec'] - dec) < tolerance)).any()
    in_vislmc = ((np.abs(lmc_vis_binaries['ra'] - RA) < tolerance) & 
                (np.abs(lmc_vis_binaries['dec'] - dec) < tolerance)).any()
    in_binary_files = "yes" if (in_optsmc or in_vissmc or in_optlmc or in_vislmc) else "no"
    
    amp = largest_amplitude(star_idx)[1]
    surveys = choose_phot.iloc[star_idx]
    surveys_used = []
    if surveys['choose_MCPS'] == 1:
        surveys_used.append('MCPS')
    if surveys['choose_APASS'] == 1:
        surveys_used.append('APASS')
    if surveys['choose_SMASH'] == 1:
        surveys_used.append('SMASH')
    surveys_used_str = ', '.join(surveys_used) if surveys_used else 'None'
    # Add annotation with both model parameters - use raw (unrounded) Av values for display
    annotation_text = ('V-redder Model: Teff={:.0f}K, Mean Teff={:.0f} $\\pm$ {:.0f}K, logL={:.2f}, logg={:.1f}, Av={:.3f}\n'
                        'B-redder Model: Teff={:.0f}K, Mean Teff={:.0f} $\\pm$ {:.0f}K, logL={:.2f}, logg={:.1f}, Av={:.3f}\n'
                        'U-redder Model: Teff={:.0f}K, Mean Teff={:.0f} $\\pm$ {:.0f}K, logL={:.2f}, logg={:.1f}, Av={:.3f}\n'
                        'Surveys used for fitting:' ' {}\n'
                        'Variability Amplitude: {:.2f} mag\n'
                        'In binary files: {}').format(
                        data_V_redder['model']['teff'], temp_stats.iloc[star_idx]['teff_mean_V_redder'], temp_stats.iloc[star_idx]['teff_std_V_redder'], data_V_redder['logL'], data_V_redder['model']['logg'], median_av_V_redder_raw,
                        data_B_redder['model']['teff'], temp_stats.iloc[star_idx]['teff_mean_B_redder'], temp_stats.iloc[star_idx]['teff_std_B_redder'], data_B_redder['logL'], data_B_redder['model']['logg'], median_av_B_redder_raw,
                        data_U_redder['model']['teff'], temp_stats.iloc[star_idx]['teff_mean_U_redder'], temp_stats.iloc[star_idx]['teff_std_U_redder'], data_U_redder['logL'], data_U_redder['model']['logg'], median_av_U_redder_raw,
                        surveys_used_str,
                        amp,
                        in_binary_files)
    
    ax.annotate(annotation_text, xy=(0.01, 0.15), xycoords='axes fraction', 
                va='top', color='black', fontsize=9, 
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    chi2_annotation = (f'$\chi^2$ Values:\n'
                       f'V-redder: {temp_stats.iloc[star_idx]["chi2_V_redder"]:.2f}\n'
                       f'B-redder: {temp_stats.iloc[star_idx]["chi2_B_redder"]:.2f}\n'
                       f'U-redder: {temp_stats.iloc[star_idx]["chi2_U_redder"]:.2f}')
    ax.annotate(chi2_annotation, xy=(0.87, 0.01), xycoords='axes fraction', 
            va='bottom', color='black', fontsize=9, 
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    

    # ax.annotate(chi2_annotation, xy=(0.82, 0.87), xycoords='axes fraction', 
    #             va='bottom', color='black', fontsize=9, 
    #             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

    ax.set_xlabel(r'Wavelength ($\AA$)')
    ax.set_ylabel(r'Flux (erg/s/cm$^{2}$/$\AA$)')
    ax.set_title(f'{RA}, {dec}; {star_idx} - Median Model Comparison: V-redder vs B-redder vs U-redder Fitting')
    ax.set_xlim(1000, 25000)
    # ax.set_xlim(1000, 12000) 
    ax.set_yscale('log')
    y_min = min(np.min(data_V_redder['scaled_model_fluxes']), np.min(data_B_redder['scaled_model_fluxes']), np.min(data_U_redder['scaled_model_fluxes']))
    # ax.set_ylim(bottom=np.log10(y_min)*1E4, top=np.log10(1))  # Set y-limits to show 4 orders of magnitude above the minimum flux
    ax.set_ylim(10**(-16), 10**(-13))
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5))
    # ax.legend(loc='upper right')
    return

# fit_models_to_star_flux(star_idx=522)

<>:413: SyntaxWarning: invalid escape sequence '\c'
<>:413: SyntaxWarning: invalid escape sequence '\c'
/var/folders/34/3847lqd14j78mfm70_z1c_r00000gn/T/ipykernel_35719/2539911469.py:413: SyntaxWarning: invalid escape sequence '\c'
  chi2_annotation = (f'$\chi^2$ Values:\n'


In [13]:
for index in range(0,50):
    star_idx = index #384
    fit_models_to_star_flux(star_idx) #522, 417, 514, 51, 220, 283, 170, 429, 748, 675, 496
    # print the number of excluded points for this star
    print(f"Number of excluded points for star {star_idx}: {temp_stats['total_exclusions'].iloc[star_idx]}, most common band excluded: {temp_stats['most_common_excluded_overall'][star_idx]}")
    # print offsets from choose_surveys_v3.csv for this star
    print(f"Offsets for star {star_idx}:")
    offset_cols = ['Offset_MCPS_APASS_4380A', 'Offset_MCPS_APASS_5450A', 
                   'Offset_SMASH_APASS_4600A', 'Offset_SMASH_APASS_5600A',
                   'Offset_MCPS_SMASH_4600A', 'Offset_MCPS_SMASH_5600A']
    for col in offset_cols:
        offset_val = choose_surveys.iloc[star_idx][col]
        if abs(offset_val) > 0.2:
            print(f"  {col}: {offset_val:.3f} mag")
    histo(star_idx)
    # info(star_idx)
    # offset_corrector(star_idx, g=True, mag_space=True, show_window=False)

/var/folders/34/3847lqd14j78mfm70_z1c_r00000gn/T/ipykernel_35719/2539911469.py:413: SyntaxWarning: invalid escape sequence '\c'
  chi2_annotation = (f'$\chi^2$ Values:\n'


FileNotFoundError: [Errno 2] No such file or directory: 'UCAC4_match.csv'